# Install and import librairies

In [3]:
import sys
import os

def install_openpyxl():
    try:
        import openpyxl
        print("openpyxl est déjà installé.")
    except ImportError:
        print("Installation de openpyxl en cours...")
        os.system(f"{sys.executable} -m pip install openpyxl")
        print("Installation terminée.")

if __name__ == "__main__":
    install_openpyxl()


openpyxl est déjà installé.


In [4]:
import os
import re
from openpyxl import Workbook


# Extract the Informations (filename and company name) to put it in the excel later

In [5]:
import os
import re

def extract_info_from_file(file_path):
    """
    Extrait les informations <FileName>...<FileName> et 'COMPANY CONFORMED NAME'
    d'un fichier texte.

    Retourne un tuple (filename_tag, company_name).
    Si une info n'est pas trouvée ou qu'une erreur survient, retourne des chaînes vides.
    """
    filename_pattern = re.compile(r"<FileName>(.*?)</FileName>")
    filename_tag = ""
    company_name = ""

    if not os.path.isfile(file_path):
        return "", ""

    try:
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            for line in f:
                match_filename = filename_pattern.search(line)
                if match_filename and not filename_tag:
                    filename_tag = match_filename.group(1).strip()
                if "COMPANY CONFORMED NAME:" in line and not company_name:
                    parts = line.split("COMPANY CONFORMED NAME:")
                    if len(parts) > 1:
                        company_name = parts[1].strip()
    except Exception as e:
        filename_tag = ""
        company_name = ""
    return filename_tag, company_name


In [9]:
data_folder = "data"
results = []
for root, dirs, files in os.walk(data_folder):
    for file_name in files:
        if file_name.lower().endswith(".txt"):
            file_path = os.path.join(root, file_name)
            filename_tag, company_name = extract_info_from_file(file_path)
            print(f"Filename: {filename_tag}")
            if filename_tag or company_name:
                results.append({
                    "filename_tag": filename_tag,
                    "company_name": company_name
                })


Filename: 20230103_10-K-A_edgar_data_1880151_0001104659-22-131423.txt
Filename: 20230103_10-K_edgar_data_1487931_0001477932-23-000012.txt
Filename: 202301z03_10-K_edgar_data_1828739_0001477932-23-000002.txt
Filename: 20230104_10-K-A_edgar_data_1889450_0001493152-23-000281.txt
Filename: 20230104_10-K_edgar_data_715446_0001493152-23-000346.txt
Filename: 20230105_10-K_edgar_data_1673431_0001096906-23-000016.txt
Filename: 20230106_10-K-A_edgar_data_1726126_0001558370-23-000085.txt
Filename: 20230106_10-K_edgar_data_1619096_0001091818-23-000002.txt
Filename: 20230106_10-K_edgar_data_1620749_0001640334-23-000027.txt
Filename: 20230106_10-K_edgar_data_315374_0001558370-23-000097.txt


# Processing the data to remove useless data

In [8]:
import os
import re

base_dir = "./data"

for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.endswith(".txt"):
            input_path = os.path.join(root, file)

            # Création du dossier de sortie
            relative_dir = os.path.relpath(root, base_dir)
            processed_dir = os.path.join(base_dir, f"processed_{relative_dir}")
            os.makedirs(processed_dir, exist_ok=True)
            output_path = os.path.join(processed_dir, file)

            try:
                with open(input_path, "r", encoding="utf-8") as f:
                    content = f.read()
            except FileNotFoundError:
                print(f"Le fichier spécifié {input_path} est introuvable. Passé.")
                continue

            lower_content = content.lower()
            start_pattern = r"item\s*1\s*a\.?\s*risk factors"
            end_pattern = r"item\s*1\s*b\.?\s*unresolved staff"

            start_match = list(re.finditer(start_pattern, lower_content))
            end_match = list(re.finditer(end_pattern, lower_content))

            if not start_match:
                print(f"Aucun 'ITEM 1A RISK FACTORS' trouvé dans {input_path}. Passé.")
                continue
            if not end_match:
                print(f"Aucun 'ITEM 1B UNRESOLVED STAFF' trouvé dans {input_path}. Passé.")
                continue

            start_idx = start_match[-1].end()
            end_idx = end_match[-1].start()

            if start_idx > end_idx:
                print(f"La section 'ITEM 1A RISK FACTORS' apparaît après 'ITEM 1B UNRESOLVED STAFF' dans {input_path}. Passé.")
                continue

            extracted_text = content[start_idx:end_idx].strip()

            with open(output_path, "w", encoding="utf-8") as out:
                out.write(extracted_text)

            print(f"Extraction terminée pour {input_path}. Résultat sauvegardé dans {output_path}.")

Extraction terminée pour ./data\QTR1,\20230103_10-K-A_edgar_data_1880151_0001104659-22-131423.txt. Résultat sauvegardé dans ./data\processed_QTR1,\20230103_10-K-A_edgar_data_1880151_0001104659-22-131423.txt.
Extraction terminée pour ./data\QTR1,\20230103_10-K_edgar_data_1487931_0001477932-23-000012.txt. Résultat sauvegardé dans ./data\processed_QTR1,\20230103_10-K_edgar_data_1487931_0001477932-23-000012.txt.
Extraction terminée pour ./data\QTR1,\20230103_10-K_edgar_data_1828739_0001477932-23-000002.txt. Résultat sauvegardé dans ./data\processed_QTR1,\20230103_10-K_edgar_data_1828739_0001477932-23-000002.txt.
Extraction terminée pour ./data\QTR1,\20230104_10-K-A_edgar_data_1889450_0001493152-23-000281.txt. Résultat sauvegardé dans ./data\processed_QTR1,\20230104_10-K-A_edgar_data_1889450_0001493152-23-000281.txt.
Extraction terminée pour ./data\QTR1,\20230104_10-K_edgar_data_715446_0001493152-23-000346.txt. Résultat sauvegardé dans ./data\processed_QTR1,\20230104_10-K_edgar_data_715446_

# Create the excel

In [7]:
def create_excel(data_list, output_filename="output.xlsx"):
    """
    Crée un fichier Excel avec trois colonnes :
    Filename, Company name, AI Probability
    """
    wb = Workbook()
    ws = wb.active
    ws.title = "Data"
    ws.append(["Filename", "Company Name", "AI Probability"])

    for item in data_list:
        filename_tag = item["filename_tag"]
        company_name = item["company_name"]
        ws.append([filename_tag, company_name, ""])

    wb.save(output_filename)
    print(f"Fichier Excel généré : {output_filename}")


In [8]:
create_excel(results, output_filename="output.xlsx")

Fichier Excel généré : output.xlsx
